In [1]:
# =============================================================
# TATA STEEL DCF VALUATION MODEL
# Data Source: Tickertape (Consolidated)
# Currency: INR Crores
# Author: [Your Name]
# =============================================================

# --- HISTORICAL DATA ---
years = ["FY2021", "FY2022", "FY2023", "FY2024", "FY2025"]

revenue =     [156294, 243959, 243353, 229171, 218543]
ebitda =      [ 27708,  63830,  32698,  23402,  25298]
depreciation =[9234,   9101,   9335,   9882,  10421]
capex =       [ 6979,  10520,  15671,  18207,  15670]



In [2]:
# --- DERIVE EBIT FROM EBITDA ---
ebit = []
for i in range(len(years)):
    ebit.append(ebitda[i] - depreciation[i])

# --- HISTORICAL MARGINS ---
ebitda_margins = []
ebit_margins = []

for i in range(len(years)):
    ebitda_margins.append(round(ebitda[i] / revenue[i] * 100, 2))
    ebit_margins.append(round(ebit[i] / revenue[i] * 100, 2))

# --- PRINT HISTORICAL TABLE ---
print(f"{'Year':<10} {'Revenue':>12} {'EBITDA':>12} {'EBIT':>12} {'EBITDA%':>10} {'EBIT%':>10}")
print("-" * 68)
for i in range(len(years)):
    print(f"{years[i]:<10} {revenue[i]:>12,} {ebitda[i]:>12,} {ebit[i]:>12,} {ebitda_margins[i]:>9}% {ebit_margins[i]:>9}%")

Year            Revenue       EBITDA         EBIT    EBITDA%      EBIT%
--------------------------------------------------------------------
FY2021          156,294       27,708       18,474     17.73%     11.82%
FY2022          243,959       63,830       54,729     26.16%     22.43%
FY2023          243,353       32,698       23,363     13.44%       9.6%
FY2024          229,171       23,402       13,520     10.21%       5.9%
FY2025          218,543       25,298       14,877     11.58%      6.81%


In [3]:
# --- PROJECTION ASSUMPTIONS ---
projection_years = ["FY2026", "FY2027", "FY2028", "FY2029", "FY2030"]
n_years = 5

revenue_growth_rate  = 0.045   # 4.5% — India infrastructure demand
ebitda_margin        = 0.125    # 12.5% — normalised, above trough, below peak
depreciation_rate    = 0.045   # 4.5% of revenue — stable asset base
maintenance_capex_rate           = 0.045    # 4.5% — moderating post expansion
tax_rate             = 0.31    # 31% — normalised effective rate
wc_change_rate       = 0.00    # 0% — neutral, conservative
# --- WACC ---
risk_free_rate       = 0.07
beta                 = 1.4
equity_risk_premium  = 0.05
cost_of_equity       = risk_free_rate + beta * equity_risk_premium

cost_of_debt         = 0.07
after_tax_cost_debt  = cost_of_debt * (1 - tax_rate)

equity_weight        = 0.725
debt_weight          = 0.275

wacc = (equity_weight * cost_of_equity) + (debt_weight * after_tax_cost_debt)

print(f"Cost of Equity:        {cost_of_equity*100:.2f}%")
print(f"After-tax Cost of Debt:{after_tax_cost_debt*100:.2f}%")
print(f"WACC:                  {wacc*100:.2f}%")

Cost of Equity:        14.00%
After-tax Cost of Debt:4.83%
WACC:                  11.48%


In [4]:
growth_capex_rates = [0.025, 0.020, 0.015, 0.010, 0.005]


def project_fcf_v2(base_revenue, n, rev_growth, ebitda_mg, dep_rate,
                   growth_capex_rates, tax_r, wc_rate):
    projected = []
    current_revenue = base_revenue
    for i in range(n):
        current_revenue = current_revenue * (1 + rev_growth)
        ebitda_val   = current_revenue * ebitda_mg
        dep_val      = current_revenue * dep_rate
        ebit_val     = ebitda_val - dep_val
        tax_val      = ebit_val * tax_r
        nopat        = ebit_val - tax_val
        growth_capex = current_revenue * growth_capex_rates[i]
        wc_val       = current_revenue * wc_rate
        fcf          = nopat - growth_capex - wc_val
        projected.append({
            "revenue":      round(current_revenue, 0),
            "ebitda":       round(ebitda_val, 0),
            "ebit":         round(ebit_val, 0),
            "growth_capex": round(growth_capex, 0),
            "fcf":          round(fcf, 0)
        })
    return projected

projections_v2 = project_fcf_v2(
    base_revenue=218543,
    n=n_years,
    rev_growth=revenue_growth_rate,
    ebitda_mg=ebitda_margin,
    dep_rate=depreciation_rate,
    growth_capex_rates=growth_capex_rates,
    tax_r=tax_rate,
    wc_rate=wc_change_rate
)

In [5]:
# --- DCF VALUATION ---
terminal_growth_rate = 0.05

# Discount FCFs
pv_fcfs = []
for i in range(n_years):
    pv = projections_v2[i]["fcf"] / (1 + wacc) ** (i + 1)
    pv_fcfs.append(round(pv, 0))

# Terminal Value
final_fcf = projections_v2[-1]["fcf"]
terminal_value = final_fcf * (1 + terminal_growth_rate) / (wacc - terminal_growth_rate)
pv_terminal_value = round(terminal_value / (1 + wacc) ** n_years, 0)

# Enterprise and Equity Value
enterprise_value = sum(pv_fcfs) + pv_terminal_value
net_debt = 94801 - 12222
equity_value = enterprise_value - net_debt
shares_outstanding = 1247

intrinsic_value = round(equity_value / shares_outstanding, 2)

print(f"Sum of PV(FCF):        {sum(pv_fcfs):>12,.0f} Crores")
print(f"PV Terminal Value:     {pv_terminal_value:>12,.0f} Crores")
print(f"Enterprise Value:      {enterprise_value:>12,.0f} Crores")
print(f"Net Debt:              {net_debt:>12,.0f} Crores")
print(f"Equity Value:          {equity_value:>12,.0f} Crores")
print(f"Intrinsic Value/Share: ₹{intrinsic_value:>10,.2f}")
print(f"Current Market Price:  ₹{200.79:>10,.2f}")
print(f"Upside/Downside:       {((intrinsic_value/200.79)-1)*100:.1f}%")

Sum of PV(FCF):              35,753 Crores
PV Terminal Value:          128,710 Crores
Enterprise Value:           164,463 Crores
Net Debt:                    83,421 Crores
Equity Value:                81,042 Crores
Intrinsic Value/Share: ₹     64.99
Current Market Price:  ₹    200.79
Upside/Downside:       -67.6%


In [ ]:
ebitda_margin_range = [0.10, 0.12, 0.14, 0.16, 0.18]
growth_rate_range = [0.03, 0.04, 0.045, 0.05, 0.06]

In [8]:
ebitda_margin_range = [0.10, 0.12, 0.14, 0.16, 0.18]
growth_rate_range = [0.03, 0.04, 0.045, 0.05, 0.06]

for margin in ebitda_margin_range:
    print(f"{margin*100:.0f}% margin", end="")

    for growth in growth_rate_range:
        proj = project_fcf_v2(
            218543, n_years, growth, margin,
            depreciation_rate, growth_capex_rates, tax_rate, wc_change_rate
        )

        pv_fcf_sum = 0
        for i in range(n_years):
            pv_fcf_sum += proj[i]["fcf"] / (1 + wacc)**(i+1)

        tv = proj[-1]["fcf"] * (1 + terminal_growth_rate) / (wacc - terminal_growth_rate)
        pv_tv = tv / (1 + wacc)**n_years

        ev = pv_fcf_sum + pv_tv
        eq_val = ev - net_debt
        iv = round(eq_val / shares_outstanding, 0)

        print(f"  ₹{iv:>6.0f}", end="")
    print()

10% margin  ₹    12  ₹    15  ₹    17  ₹    19  ₹    23
12% margin  ₹    48  ₹    53  ₹    55  ₹    58  ₹    64
14% margin  ₹    84  ₹    90  ₹    94  ₹    97  ₹   105
16% margin  ₹   120  ₹   128  ₹   132  ₹   137  ₹   145
18% margin  ₹   156  ₹   165  ₹   171  ₹   176  ₹   186
